In [61]:
cameo_codes = {
    "MAKE PUBLIC STATEMENT": "1",
    "APPEAL": "2",
    "EXPRESS INTENT TO COOPERATE": "3",
    "CONSULT": "4",
    "ENGAGE IN DIPLOMATIC COOPERATION": "5",
    "ENGAGE IN MATERIAL COOPERATION": "6",
    "PROVIDE AID": "7",
    "YIELD": "8",
    "INVESTIGATE": "9",
    "DEMAND": "10",
    "DISAPPROVE": "11",
    "REJECT": "12",
    "THREATEN": "13",
    "PROTEST": "14",
    "EXHIBIT FORCE POSTURE": "15",
    "COERCE": "16",
    "ASSAULT": "17",
    "FIGHT": "19",
    "USE UNCONVENTIONAL MASS VIOLENCE": "20",
}

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.express as px
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import re

import logging
logger = logging.getLogger(__name__)

In [2]:
from e_embedding import *

In [ ]:
checkpoint = torch.load('/data/jupyter/con_models/contrastive_64_1.pt')

model = ContrastiveLossEmbeddingModel(
    checkpoint["input_dim"],
    checkpoint["embed_dim"]
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

ContrastiveLossEmbeddingModel(
  (embedding): Linear(in_features=384, out_features=384, bias=True)
)

In [ ]:


def vectorize(series):
    """
    Robust parser for embedding_json that handles:
    - space or comma separated floats
    - newlines
    - brackets or no brackets
    Returns: np.ndarray of shape (N, D)
    """
    vecs = []

    for s in series.astype(str):
        # Remove brackets and newlines
        s = s.replace("[", "").replace("]", "").replace("\n", " ").strip()

        # Split on whitespace OR comma
        parts = re.split(r"[,\s]+", s)

        # Convert to float
        vec = np.array(parts, dtype=np.float32)
        vecs.append(vec)

    return np.vstack(vecs)


In [ ]:
global_db = pd.read_csv('/data/elugos/global_db_111125.csv')
df_val_sample = pd.read_csv(Path('/data/elugos/val_data.csv'))

In [ ]:
X_new = vectorize(df_val_sample["embedding_json"])   # shape = (N, 384)
X_new = torch.from_numpy(X_new).float()

In [142]:
with torch.no_grad():
    new_embeddings = model(X_new).cpu().numpy()   # shape = (N, 64)

In [ ]:
# first step grabs the intersection of global_db and val data

learned_test_df = df_val_sample.merge(
    global_db[["GlobalEventID", "EventCode", "title"]],
    on="GlobalEventID",
    how="left"
)


In [ ]:
learned_test_df = learned_test_df[[
    "GlobalEventID",
    "EventCode",
    "title"
]].copy()
#append learned embeddings to new df
learned_test_df["learned_embedding"] = list(new_embeddings)


In [26]:
def make_sampler(df,num):
    # sample i and j
    i = df.sample(n=num, random_state=42)
    remaining = df.drop(i.index)
    j = remaining.sample(n=num, random_state=43)

    # reset index so pairs line up
    i = i.reset_index(drop=True)
    j = j.reset_index(drop=True)

    # Extract only the FIRST part of EventCode (before underscore)
    i_event_major = i["EventCode"].astype(str).str.split("_").str[-1]
    j_event_major = j["EventCode"].astype(str).str.split("_").str[-1]

    # Build sampler
    sampler = pd.DataFrame({
        "GlobalEventID-i": i["GlobalEventID"],
        "GlobalEventID-j": j["GlobalEventID"],
        "embedding_json-i": i["learned_embedding"],
        "embedding_json-j": j["learned_embedding"],
        "EventCode-i": i_event_major,
        "EventCode-j": j_event_major
    })

    # y = 1 if the FIRST XX match, else 0
    sampler["y"] = (i_event_major.values == j_event_major.values).astype(int)

    return sampler


In [145]:
sampler = make_sampler(learned_test_df,20000)

In [146]:
i_np = vectorize(sampler["embedding_json-i"])
j_np = vectorize(sampler["embedding_json-j"])

i = torch.from_numpy(i_np)
j = torch.from_numpy(j_np)
# Random labels: 0 if dissimilar, 1 if similar.
#make the labels if similar 1, 0 if not
labels = torch.tensor(sampler["y"].values, dtype=torch.float32)

In [155]:
sampler = make_sampler(learned_test_df,20000)

In [ ]:
def avg_pairwise_distance_by_label(x1, x2, labels, metric="cosine"):
    """
    Works with BOTH numpy arrays and torch tensors.
    Computes avg distance for same-label and opposite-label pairs.
    """

    # --- Convert everything to torch tensors if needed ---
    if isinstance(x1, np.ndarray):
        x1 = torch.from_numpy(x1)
    if isinstance(x2, np.ndarray):
        x2 = torch.from_numpy(x2)
    if isinstance(labels, np.ndarray):
        labels = torch.from_numpy(labels)

    x1 = x1.float()
    x2 = x2.float()
    labels = labels.int()

    # --- Distance computation ---
    if metric == "cosine":
        x1n = F.normalize(x1, p=2, dim=1)
        x2n = F.normalize(x2, p=2, dim=1)
        distances = 1 - torch.sum(x1n * x2n, dim=1)

    elif metric == "euclidean":
        distances = torch.norm(x1 - x2, dim=1)

    else:
        raise ValueError("metric must be 'cosine' or 'euclidean'")

    distances = distances.cpu().numpy()
    labels = labels.cpu().numpy()

    same_mask = labels == 1
    opp_mask  = labels == 0

    same_dist = distances[same_mask]
    opp_dist = distances[opp_mask]

    avg_same = distances[same_mask].mean() if same_mask.any() else np.nan
    avg_opp  = distances[opp_mask].mean() if opp_mask.any() else np.nan

    return same_dist, opp_dist,{
        "avg_same": float(avg_same),
        "avg_opposite": float(avg_opp),
        "n_same": int(same_mask.sum()),
        "n_opposite": int(opp_mask.sum())
    }

In [ ]:
# SBERT baseline


sbert_same_dist, sbert_opp_dist, stats_sbert_35000 = avg_pairwise_distance_by_label(i, j, labels, metric="cosine")
#learned_same_dist, learned_opp_dist, learned_sbert_35000 = avg_pairwise_distance_by_label(i, j, labels, metric="cosine")
# Learned embeddings
#stats_learne = avg_pairwise_distance_by_label(i_embed, j_embed, labels, metric="euclidean")

In [148]:
stats_sbert_35000

{'avg_same': 0.9948474168777466,
 'avg_opposite': 0.9993056058883667,
 'n_same': 2103,
 'n_opposite': 17897}

In [ ]:
np.savetxt('model_same_cos_64_1.csv', sbert_same_dist, delimiter=',')
np.savetxt('model_opp_cos_64_1.csv', sbert_opp_dist, delimiter=',')
